In [2]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"
start = NY_tz.localize(datetime.datetime(2025, 12, 29, 7, 0))
end = NY_tz.localize(datetime.datetime(2025, 12, 29, 17, 30))

In [4]:
from SDRUtils.products import USD_SOFR_SwapProduct

product = USD_SOFR_SwapProduct()
df = product.build_classification_dataframe(start=start, end=end, cache_path=cache_path)

df

CACHE HIT...: 100%|██████████| 1/1 [00:00<00:00, 26.47it/s]


,trade_id,execution_timestamp,effective_date,expiration_date,product_type,tenor_years,tenor_label,is_forward,forward_start_years,forward_label,...,Cleared,package_legs,matched_ust_maturity,ust_cusip,ust_oi,ust_issue_date,swap_maturity_date,matched_ust_maturity_trade_confidence,invoice_swap_ticker,risk
0,1570142946000000101,2025-12-29 12:03:13+00:00,2026-08-06,2036-08-06 00:00:00,OIS_SWAP,10.147222,10Y,True,0.611111,7M,...,I,None,False,None,None,None,2036-08-06,NaN,None,2500.0
1,1570299802000000201,2025-12-29 12:05:58+00:00,2026-03-18,2046-03-18 00:00:00,OIS_SWAP,20.294444,IMM_H2046,True,0.219444,IMM_H2026,...,I,None,False,None,None,None,2046-03-18,NaN,None,2500.0
2,1570469108000000701,2025-12-29 12:09:34+00:00,2026-03-18,2051-03-18 00:00:00,OIS_SWAP,25.369444,IMM_H2051,True,0.219444,IMM_H2026,...,I,None,False,None,None,None,2051-03-18,NaN,None,2500.0
3,1570184451000000301,2025-12-29 12:09:46+00:00,2026-03-18,2031-03-18 00:00:00,OIS_SWAP,5.072222,IMM_H2031,True,0.219444,IMM_H2026,...,I,None,False,None,None,None,2031-03-18,NaN,None,17500.0
4,1570196242000000201,2025-12-29 12:11:05+00:00,2026-03-18,2036-03-18 00:00:00,OIS_SWAP,10.147222,IMM_H2036,True,0.219444,IMM_H2026,...,I,None,False,None,None,None,2036-03-18,NaN,None,10000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1605,1573649665000000201,2025-12-29 21:53:50+00:00,2025-12-31,2029-12-31 00:00:00,OIS_SWAP,4.058333,4Y,False,0.005556,spot,...,I,[1573649665000000201],True,91282CMD0,5-Year,2024-12-31,2029-12-31,low,NaN,2500.0
1606,1573656731000000101,2025-12-29 21:54:42+00:00,2025-12-31,2026-12-31 00:00:00,OIS_SWAP,1.013889,1Y,False,0.005556,spot,...,I,[1573656731000000101],True,91282CME8,2-Year,2024-12-31,2026-12-31,low,NaN,5000.0
1607,1573691775000000101,2025-12-29 21:59:26+00:00,2025-12-31,2032-12-31 00:00:00,OIS_SWAP,7.102778,7Y,False,0.005556,spot,...,I,[1573691775000000101],True,91282CPQ8,7-Year,2025-12-31,2032-12-31,low,NaN,2500.0
1608,1573723141000000201,2025-12-29 21:52:38+00:00,2025-12-31,2055-11-15 00:00:00,OIS_SWAP,30.308333,30Y,False,0.005556,spot,...,I,[1573723141000000201],True,912810UP1,30-Year,2025-11-17,2055-11-15,high,NaN,100000.0


In [5]:
df[df["invoice_swap_ticker"] == "TYA"]

,trade_id,execution_timestamp,effective_date,expiration_date,product_type,tenor_years,tenor_label,is_forward,forward_start_years,forward_label,...,Cleared,package_legs,matched_ust_maturity,ust_cusip,ust_oi,ust_issue_date,swap_maturity_date,matched_ust_maturity_trade_confidence,invoice_swap_ticker,risk
978,1570476216000000301,2025-12-29 12:43:06+00:00,2026-03-31,2032-11-15 00:00:00,OIS_SWAP,6.725,7Y,True,0.255556,3M,...,I,[1570476216000000301],True,91282CFV8,10-Year,2022-11-15,2032-11-15,high,TYA,25221.346128
984,1570638065000001001,2025-12-29 12:54:12+00:00,2026-03-31,2032-11-15 00:00:00,OIS_SWAP,6.725,7Y,True,0.255556,3M,...,I,[1570638065000001001],True,91282CFV8,10-Year,2022-11-15,2032-11-15,high,TYA,99712.298644
991,1570702409000000101,2025-12-29 12:56:44+00:00,2026-03-31,2032-11-15 00:00:00,OIS_SWAP,6.725,7Y,True,0.255556,3M,...,N,[1570702409000000101],True,91282CFV8,10-Year,2022-11-15,2032-11-15,high,TYA,93846.869312
1023,1570769153000000601 / 1570770130000000201,2025-12-29T13:33:53+00:00 / 2025-12-29T13:33:5...,2026-03-31,2032-11-15T00:00:00 / 2052-11-15T00:00:00,OIS_SWAP,6.725 / 27.0167,7Y / 27Y,True,0.255556,3M,...,I,"[1570769153000000601, 1570770130000000201]",True,91282CFV8 / 912810TL2,10-Year / 30-Year,2022-11-15,2032-11-15 / 2052-11-15,high,TYA,39884.900000
1030,1570742278000001101,2025-12-29 13:40:38+00:00,2026-03-31,2032-11-15 00:00:00,OIS_SWAP,6.725,7Y,True,0.255556,3M,...,I,[1570742278000001101],True,91282CFV8,10-Year,2022-11-15,2032-11-15,high,TYA,175.962880
1085,1570792823000001201 / 1570792827000001601,2025-12-29 14:04:28+00:00,2026-03-31,2032-11-15T00:00:00 / 2052-11-15T00:00:00,OIS_SWAP,6.725 / 27.0167,7Y / 27Y,True,0.255556,3M,...,I,"[1570792823000001201, 1570792827000001601]",True,91282CFV8 / 912810TL2,10-Year / 30-Year,2022-11-15,2032-11-15 / 2052-11-15,high,TYA,39884.900000
1142,1570877289000000401,2025-12-29 14:51:54+00:00,2026-03-31,2032-11-15 00:00:00,OIS_SWAP,6.725,7Y,True,0.255556,3M,...,I,[1570877289000000401],True,91282CFV8,10-Year,2022-11-15,2032-11-15,high,TYA,9971.229864
1161,1570939161000000201,2025-12-29 15:09:18+00:00,2026-03-31,2032-11-15 00:00:00,OIS_SWAP,6.725,7Y,True,0.255556,3M,...,I,[1570939161000000201],True,91282CFV8,10-Year,2022-11-15,2032-11-15,high,TYA,134904.874636
1164,1570916630000000101,2025-12-29 15:10:19+00:00,2026-03-31,2032-11-15 00:00:00,OIS_SWAP,6.725,7Y,True,0.255556,3M,...,I,[1570916630000000101],True,91282CFV8,10-Year,2022-11-15,2032-11-15,high,TYA,49856.149322
1187,1570923934000000801 / 1570923935000000901,2025-12-29 15:25:58+00:00,2026-03-31,2031-12-31T00:00:00 / 2032-11-15T00:00:00,OIS_SWAP,5.83611 / 6.725,6Y / 7Y,True,0.255556,3M,...,I,"[1570923935000000901, 1570923934000000801]",True,91282CMC2 / 91282CFV8,7-Year / 10-Year,2024-12-31 / 2022-11-15,2031-12-31 / 2032-11-15,high,TYA,103398.000000


In [184]:
# df.tail(1).to_dict(orient="records")
df[(df["effective_date"].dt.date == datetime.date(2026, 3, 31))]

,trade_id,execution_timestamp,effective_date,expiration_date,product_type,tenor_years,tenor_label,is_forward,forward_start_years,forward_label,...,Cleared,package_legs,matched_ust_maturity,ust_cusip,ust_oi,ust_issue_date,swap_maturity_date,matched_ust_maturity_trade_confidence,invoice_swap_ticker,risk
974,1570380985000000401,2025-12-29 12:38:22+00:00,2026-03-31,2026-12-31 00:00:00,OIS_SWAP,0.763889,9M,True,0.255556,3M,...,I,[1570380985000000401],True,91282CME8,2-Year,2024-12-31,2026-12-31,high,NaN,5533.409236
978,1570476216000000301,2025-12-29 12:43:06+00:00,2026-03-31,2032-11-15 00:00:00,OIS_SWAP,6.725,7Y,True,0.255556,3M,...,I,[1570476216000000301],True,91282CFV8,10-Year,2022-11-15,2032-11-15,high,TYA,25221.346128
982,1570527359000000301,2025-12-29 12:45:25+00:00,2026-03-31,2032-11-30 00:00:00,OIS_SWAP,6.766667,7Y,True,0.255556,3M,...,I,[1570527359000000301],True,91282CPM7,7-Year,2025-12-01,2032-11-30,high,TYB,100245.298930
984,1570638065000001001,2025-12-29 12:54:12+00:00,2026-03-31,2032-11-15 00:00:00,OIS_SWAP,6.725,7Y,True,0.255556,3M,...,I,[1570638065000001001],True,91282CFV8,10-Year,2022-11-15,2032-11-15,high,TYA,99712.298644
990,1570527354000000201 / 1570531687000000101,2025-12-29T12:56:38+00:00 / 2025-12-29T12:57:0...,2026-03-31,2035-08-15T00:00:00 / 2043-11-15T00:00:00,OIS_SWAP,9.51111 / 17.8861,10Y / 18Y,True,0.255556,3M,...,I,"[1570527354000000201, 1570531687000000101]",True,91282CNT4 / 912810TW8,10-Year / 20-Year,2025-08-15 / 2023-11-30,2035-08-15 / 2043-11-15,high,UTA,49656.500000
991,1570702409000000101,2025-12-29 12:56:44+00:00,2026-03-31,2032-11-15 00:00:00,OIS_SWAP,6.725,7Y,True,0.255556,3M,...,N,[1570702409000000101],True,91282CFV8,10-Year,2022-11-15,2032-11-15,high,TYA,93846.869312
1002,1570768102000000101,2025-12-29 13:08:52+00:00,2026-03-31,2035-08-15 00:00:00,OIS_SWAP,9.511111,10Y,True,0.255556,3M,...,I,[1570768102000000101],True,91282CNT4,10-Year,2025-08-15,2035-08-15,high,NaN,60691.324413
1023,1570769153000000601 / 1570770130000000201,2025-12-29T13:33:53+00:00 / 2025-12-29T13:33:5...,2026-03-31,2032-11-15T00:00:00 / 2052-11-15T00:00:00,OIS_SWAP,6.725 / 27.0167,7Y / 27Y,True,0.255556,3M,...,I,"[1570769153000000601, 1570770130000000201]",True,91282CFV8 / 912810TL2,10-Year / 30-Year,2022-11-15,2032-11-15 / 2052-11-15,high,TYA,39884.900000
1030,1570742278000001101,2025-12-29 13:40:38+00:00,2026-03-31,2032-11-15 00:00:00,OIS_SWAP,6.725,7Y,True,0.255556,3M,...,I,[1570742278000001101],True,91282CFV8,10-Year,2022-11-15,2032-11-15,high,TYA,175.962880
1040,1570774051000000101,2025-12-29 13:42:46+00:00,2026-03-31,2035-08-15 00:00:00,OIS_SWAP,9.511111,10Y,True,0.255556,3M,...,I,[1570774051000000101],True,91282CNT4,10-Year,2025-08-15,2035-08-15,high,NaN,50444.737174


In [5]:
from MDP.FixedRateBonds.TRADINGVIEW.TradingViewFetcher import fetch_cusip_prices_eod_timeseries_parallel

cusips = ["91282CPD7"]
fetch_cusip_prices_eod_timeseries_parallel(cusips=cusips, start=datetime.date(2025, 12, 25), end=datetime.date(2025, 12, 29))

FETCHING CUSIPS...: 100%|██████████| 1/1 [00:01<00:00,  1.22s/it]


,91282CPD7
2025-12-26,99.6172
2025-12-29,99.8125
